In [ ]:
import pandas as pd

FILE_PATH = "data/Sale Order History.csv"

df = pd.read_csv(FILE_PATH)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Change to datetime format
DATE_FORMAT = "%m/%d/%Y"

df['Sales Order Date'] = pd.to_datetime(df['Sales Order Date'], format=DATE_FORMAT, errors="coerce")

print(f"Total missing value in Sales Order Date column: {df['Sales Order Date'].isna().sum():,}")
print(f"Sales Order Date dtype check: {df['Sales Order Date'].dtypes}")

In [ ]:
# Captitalizing all letters in Sales Order Status column
df['Sales Order Status'] = df['Sales Order Status'].str.upper()

df['Sales Order Status'].head(10)

In [ ]:
# Strip leading/trailing whitespace from every text column.
# NOTE: the previous version called .str.strip() but never assigned the
# result back, so it did nothing. This writes it back into df.
str_cols = df.select_dtypes(["object", "str"]).columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

df[str_cols].head(10)

In [ ]:
# Strip whitespace from column names too. The raw export ships
# "Customer Internal ID " and "Item Internal ID " with a trailing space,
# which silently breaks exact-name lookups. .str.strip() fixes any such
# column without having to name them one by one.
df.columns = df.columns.str.strip()

list(df.columns)

In [ ]:
# Check for missing value
df.isna().sum()

In [ ]:
# Check for duplicated rows
duplicated_before = df.duplicated().sum()

print(f"Total duplicated rows before: {duplicated_before:,}")

In [ ]:
# Get rid of duplicated rows
df = df.drop_duplicates()

print(f"Total rows after dropping duplicates: {len(df):,}")

In [ ]:
# Check for 0 value rows
(df == 0).sum()

In [ ]:
# Drop 0 value Quantity rows
df = df[df['Quantity'] != 0]

(df == 0).sum()

In [ ]:
# --- Impute zero Base Price rows --------------------------------------------
# ~4,300 line items have Base Price = 0. Those items span roughly THB 1k to
# THB 2.8M, so a single global median (THB 9,726) would badly distort revenue
# for the expensive ones. Instead we impute, in order of preference:
#   1. the item's official List price from the item master (covers ~99.9%)
#   2. that item's own median price from its non-zero rows
#   3. the global median, only as a last resort
# Revenue = Base Price x Quantity is a project deliverable, so this matters;
# it has no effect on the demand/units forecast.

item_master = pd.read_excel("data/Item_master.xlsx")
item_master.columns = item_master.columns.str.strip()
list_price = item_master.set_index("Item Internal ID")["List price"]

nonzero = df.loc[df["Base Price"] > 0]
item_median = nonzero.groupby("Item Internal ID")["Base Price"].median()
global_median = nonzero["Base Price"].median()

zero_mask = df["Base Price"] == 0
ids = df.loc[zero_mask, "Item Internal ID"]
via_lp = ids.map(list_price)
via_median = ids.map(item_median)
df.loc[zero_mask, "Base Price"] = via_lp.fillna(via_median).fillna(global_median).to_numpy()

print(f"Zero-price rows imputed: {zero_mask.sum():,}")
print(f"  via item-master list price: {via_lp.notna().sum():,}")
print(f"  via item's own median:      {(via_lp.isna() & via_median.notna()).sum():,}")
print(f"  via global median:          {(via_lp.isna() & via_median.isna()).sum():,}")
print(f"Remaining zero Base Price rows: {(df['Base Price'] == 0).sum()}")
(df == 0).sum()

In [ ]:
# Export. index=False so we don't write pandas' row index as a stray
# "Unnamed: 0" column that every downstream reader then has to ignore.
df.to_csv("data/cleaned_data.csv", index=False)

print(f"Exported {len(df):,} rows x {df.shape[1]} columns -> cleaned_data.csv")
df.head()